# Fitting an orbit with layup

[layup](https://github.com/Smithsonian/layup) is an open-source package for small-body orbit determination. This notebook walks through the full Python API end-to-end:

1. load astrometric observations,
2. fit an orbit (initial orbit determination + a Levenberg–Marquardt refinement),
3. inspect the fitted state and fit quality,
4. convert the orbit to Keplerian / cometary elements, and
5. predict future sky positions with their uncertainty.

> **Ephemeris note.** The fit integrates the orbit with [ASSIST](https://github.com/matthewholman/assist), which needs JPL planetary + small-body ephemeris kernels (a few hundred MB). Run `layup bootstrap` once to download them, or let layup fetch them into its cache on first use. Pass `cache_dir=` to point at where the kernels live; `cache_dir=None` uses layup's default cache.

In [1]:
from types import SimpleNamespace

import numpy as np
import pandas as pd

from layup.utilities.file_io.CSVReader import CSVDataReader
from layup.orbitfit import orbitfit
from layup.convert import convert
from layup.predict import predict

        Use pytest instead. [astropy.tests.runner]
        Use pytest instead. [astropy.utils.decorators]


## 1. Load observations

`example_observations.csv` holds 587 astrometric observations (MPC ADES columns) of the main-belt asteroid **(119839) 2002 CX17**. layup's readers return a structured NumPy array, one row per observation.

In [2]:
obs = CSVDataReader("example_observations.csv", "csv", primary_id_column_name="provID").read_rows()
print(f"{len(obs)} observations of object {obs['provID'][0]}")
pd.DataFrame(obs[["provID", "ra", "dec", "obsTime", "stn"]][:5])

587 observations of object 119839


,provID,ra,dec,obsTime,stn
0,119839,138.93912,17.23753,1997-03-04T05:06:21.600Z,704
1,119839,138.91971,17.23875,1997-03-04T07:43:35.616Z,704
2,119839,327.33458,-13.43625,1999-09-03T05:07:08.256Z,691
3,119839,327.32996,-13.43683,1999-09-03T05:40:26.688Z,691
4,119839,327.32442,-13.43750,1999-09-03T06:19:33.312Z,691


## 2. Fit the orbit

`orbitfit` does it all: it runs initial orbit determination (Gauss's method by default) to get a starting guess, then refines with a Levenberg–Marquardt fit whose model is an ASSIST n-body integration. It accepts many objects at once (keyed by `provID`) and returns one fitted orbit per object.

In [3]:
result = orbitfit(obs, cache_dir=None, primary_id_column_name="provID")
r = result[0]
print("convergence flag:", r["flag"], "(0 = success)")
print(f"chi-square {r['csq']:.1f} over {r['ndof']} dof " f"(reduced chi-square {r['csq'] / r['ndof']:.3f})")
print("method:", r["method"], "| output format:", r["FORMAT"])

        Use pytest instead. [astropy.tests.runner]
        Use pytest instead. [astropy.utils.decorators]


convergence flag: 0 (0 = success)
chi-square 282.1 over 1168 dof (reduced chi-square 0.242)
method: orbit_fit | output format: BCART_EQ


## 3. The fitted state

The result is reported in barycentric **equatorial Cartesian** coordinates (`BCART_EQ`): position in au and velocity in au/day, at the fit epoch (MJD, TDB). A reduced chi-square near 1 indicates a good fit given the assumed astrometric uncertainties.

In [4]:
state = np.array([r[c] for c in ("x", "y", "z", "xdot", "ydot", "zdot")])
print("position (au): ", np.round(state[:3], 6))
print("velocity (au/d):", np.round(state[3:], 6))
print("epoch (MJD TDB):", r["epochMJD_TDB"])

position (au):  [1.344744 2.44267  1.527067]
velocity (au/d): [-0.008399  0.003798  0.001567]
epoch (MJD TDB): 59546.25963587081


## 4. Convert to orbital elements

`convert` transforms a fitted orbit between representations. Here we ask for Keplerian elements (`KEP`); cometary (`COM`), barycentric variants (`BKEP`, `BCOM`), and Cartesian forms are also available. Angles are returned in degrees.

In [5]:
kep = convert(result, "KEP", primary_id_column_name="provID")[0]
labels = [("a", "au"), ("e", ""), ("inc", "deg"), ("node", "deg"), ("argPeri", "deg"), ("ma", "deg")]
for name, unit in labels:
    print(f"  {name:8s} = {kep[name]:12.5f} {unit}")

  a        =      2.99784 au
  e        =      0.06152 
  inc      =      7.77759 deg
  node     =    330.58061 deg
  argPeri  =    283.39213 deg
  ma       =    169.34010 deg


## 5. Predict future sky positions

`predict` propagates the fitted orbit — carrying its covariance — to chosen epochs and an observatory, returning RA/Dec and the on-sky 1-sigma error ellipse. We use observatory code `500` (geocenter) and three epochs (JD, TDB).

In [6]:
times = [2460676.5, 2460706.5, 2460736.5]  # JD TDB
args = SimpleNamespace(onsky_data=False, primary_id_column_name="provID")
eph = predict(
    result,
    obscode="500",
    times=times,
    num_workers=1,
    cache_dir=None,
    primary_id_column_name="provID",
    args=args,
)
for row in eph:
    print(
        f"JD {row['epoch_JD_TDB']:.1f}:  RA={row['ra_deg']:8.4f} deg  "
        f"Dec={row['dec_deg']:8.4f} deg  "
        f"(1-sigma ellipse {row['a_arcsec']:.2f} x {row['b_arcsec']:.2f} arcsec)"
    )

        Use pytest instead. [astropy.tests.runner]
        Use pytest instead. [astropy.utils.decorators]


JD 2460676.5:  RA=282.2069 deg  Dec=-27.3870 deg  (1-sigma ellipse 0.07 x 0.04 arcsec)
JD 2460706.5:  RA=295.9813 deg  Dec=-25.3899 deg  (1-sigma ellipse 0.07 x 0.04 arcsec)
JD 2460736.5:  RA=308.7342 deg  Dec=-22.5309 deg  (1-sigma ellipse 0.07 x 0.04 arcsec)


## Recap and next steps

In this notebook we loaded an observation file, ran a full orbit fit, converted the result to Cometary and Keplerian elements, and predicted future sky positions with uncertainties — the core `load → orbitfit → convert → predict` workflow.

For real surveys, `orbitfit` exposes a few more options worth knowing:

- `iod="auto"` — fall back to BK-IOD when Gauss's method struggles (short arcs).
- `engine="bk"` — fit in the Bernstein–Khushalani basis (robust for distant / poorly-constrained objects).
- `weight_data=True`, `debias=True` — apply per-observation weighting and star-catalog debiasing.
- `fit_nongrav="A2"` — additionally fit the transverse (Yarkovsky) non-gravitational parameter.
- `num_workers=N` — fit many objects in parallel.

Each of these will be covered in more detail as the [layup documentation](https://github.com/Smithsonian/layup) grows; for now, the docstrings (e.g. `help(orbitfit)`) are the full reference.